# Notebook 4 — Full Measurement Run
The complete experiment, in your required order:

1. **Mode first** (decides motor count) → motor bring-up (all gated)
2. Camera open/configure/arm, optional dark frame
3. *(4×4 only)* park both polarizers at the fixed optical angle
4. Enter PSG/PSA angle specs → preview grid → final confirmation
5. Discrete loop: move → settle 1 → trigger → save+verify BMP → log →
   **checkpoint** → settle 2 → next. Crash? Re-run this notebook and it
   offers to **resume** where it stopped.

Continuous 4×4 mode is intentionally not active yet (image-timing rule still to be decided).

In [ ]:
# Cell 1 -- mode selection + motor bring-up (identical to Notebook 1)
import sys; sys.path.insert(0, ".")
from mmie.motors import MotorBank
print("Modes:  1 = 3x3    2 = 4x4 discrete    3 = 4x4 continuous (not yet active)")
choice = input("Select mode [1/2/3]: ").strip()
mode = {"1": "3x3", "2": "4x4_discrete", "3": "4x4_continuous"}[choice]
bank = MotorBank(mode)                               # only the motors this mode needs
bank.full_bringup()                                  # gated: connect->init->enable->home->zero

In [ ]:
# Cell 2 -- camera up + optional dark frame
from mmie.camera import IDSCamera
cam = IDSCamera()                                    # wrapper object
cam.open(); cam.configure(); cam.start()             # full camera chain, armed
use_dark = input("Capture a master dark frame for subtraction? [y/N]: ").strip().lower() in ("y","yes")
if use_dark:
    cam.capture_dark_frame()                         # guided cover/uncover procedure

In [ ]:
# Cell 3 -- (4x4 modes only) park the polarizers at their fixed optical angle
from mmie.measure import park_polarizers_for_4x4
if mode.startswith("4x4"):                           # skip entirely for 3x3
    park_polarizers_for_4x4(bank)                    # gated single move for each polarizer

In [ ]:
# Cell 4 -- angle specs for the STEPPING elements, then run the measurement
from mmie.angles import parse_angle_spec
from mmie.measure import run_discrete, run_continuous
if mode == "4x4_continuous":                                     # continuous branch
    run_continuous(bank, cam)                                    # raises NotImplementedError (by design)
else:                                                            # 3x3 or 4x4_discrete
    elem = "polarizer" if mode == "3x3" else "QWP"               # what is being stepped
    psg = parse_angle_spec(input(f"PSG {elem} angle spec (e.g. 360/10 or 0,30,60): "))
    psa = parse_angle_spec(input(f"PSA {elem} angle spec (may differ):            "))
    folder = run_discrete(bank, cam, psg, psa, use_dark=use_dark)  # THE measurement loop
    print("Finished. Data in:", folder)

In [ ]:
# Cell 5 -- ALWAYS run this at the end (or after a crash) to free hardware
try:    cam.close()                                  # release the camera
except Exception as e: print("cam close:", e)
bank.shutdown_all()                                  # release all motors